# 04 Vector Database Systems: Qdrant, Pinecone & Pgvector Architectures

Implementing metadata pre-filtering vs post-filtering simulations and vector DB payload query engines.

## Setup: Repository-Level Dotenv Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load repo-level .env file
load_dotenv()

print("Phase 3 Vector DB Environment Loaded.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: Vector DB Metadata Pre-Filtering Engine

In [ ]:
# Vector Database Metadata Pre-Filtering Simulation
class VectorDBPayloadIndex:
    def __init__(self):
        self.records = []

    def insert(self, vector_id: int, vector: np.ndarray, metadata: dict):
        self.records.append({"id": vector_id, "vector": vector, "metadata": metadata})

    def query_with_prefilter(self, query_vector: np.ndarray, filter_key: str, filter_val: str, top_k: int = 2):
        # Pre-filtering: Filter dataset BEFORE computing vector distance
        filtered = [r for r in self.records if r["metadata"].get(filter_key) == filter_val]
        
        scores = []
        for r in filtered:
            sim = np.dot(query_vector, r["vector"])
            scores.append((sim, r))
        scores.sort(key=lambda x: x[0], reverse=True)
        return [item[1] for item in scores[:top_k]]

db = VectorDBPayloadIndex()
db.insert(1, np.array([0.9, 0.1]), {"tenant_id": "org_A", "category": "finance"})
db.insert(2, np.array([0.1, 0.8]), {"tenant_id": "org_B", "category": "finance"})

results = db.query_with_prefilter(np.array([0.85, 0.15]), filter_key="tenant_id", filter_val="org_A")
print(f"Pre-filtered Search Results (Tenant org_A only): {results}")